In [82]:
import pandas as pd
import numpy as np
import random
import sklearn.manifold as manifold
import colorsys

import plotly.express as px

In [83]:
# Read distance matrix
dist_mat = pd.read_csv("../output/handshape_distance_matrix.csv")

dist_mat.head()

,Name,fbx\M20241107_6263_260710_2.fbx_segment_0,fbx\M20241107_6263_260710_2.fbx_segment_1,fbx\M20241107_6263_260710_2.fbx_segment_2,fbx\M20241107_6263_260710_2.fbx_segment_3,fbx\M20250916_4405_260709_0.fbx_segment_0,fbx\M20260119_1078_260713_1.fbx_segment_0,fbx\M20260119_1078_260713_1.fbx_segment_1,fbx\M20260119_1078_260713_1.fbx_segment_2,fbx\M20260119_1078_260713_1.fbx_segment_3,...,../HandformSession/HandformsTrimmed/Q.FBX,../HandformSession/HandformsTrimmed/R.FBX,../HandformSession/HandformsTrimmed/S.FBX,../HandformSession/HandformsTrimmed/Shh.FBX,../HandformSession/HandformsTrimmed/T.FBX,../HandformSession/HandformsTrimmed/T_Open.FBX,../HandformSession/HandformsTrimmed/V.FBX,../HandformSession/HandformsTrimmed/V_Curved.FBX,../HandformSession/HandformsTrimmed/W.FBX,../HandformSession/HandformsTrimmed/Y.FBX
0,fbx\M20241107_6263_260710_2.fbx_segment_0,0.00000,6.87203,7.53916,6.20351,8.22305,9.29600,6.12951,8.79348,8.82916,...,8.62910,9.76328,11.08110,9.67635,9.23426,9.73143,8.05657,9.66211,9.55285,7.52990
1,fbx\M20241107_6263_260710_2.fbx_segment_1,6.87203,0.00000,6.76421,7.02312,7.64752,7.63650,8.13369,9.17794,10.23440,...,9.05728,7.98189,9.47957,9.34567,7.43699,8.04174,7.48038,8.98014,6.73075,7.04357
2,fbx\M20241107_6263_260710_2.fbx_segment_2,7.53916,6.76421,0.00000,7.63080,6.87088,6.61488,6.28928,8.85161,7.65659,...,9.54438,8.52185,9.06012,8.38182,7.80774,7.10852,5.26154,10.52030,7.31328,7.37346
3,fbx\M20241107_6263_260710_2.fbx_segment_3,6.20351,7.02312,7.63080,0.00000,8.30138,6.95901,7.63074,7.56932,8.81531,...,8.57687,9.71738,10.09970,8.72091,9.24255,8.65906,6.82697,9.61417,8.65546,6.21535
4,fbx\M20250916_4405_260709_0.fbx_segment_0,8.22305,7.64752,6.87088,8.30138,0.00000,7.66301,6.78347,8.03735,9.26221,...,9.18368,8.12807,8.56903,8.20912,7.54277,6.82789,7.60507,10.22120,8.21347,6.89011


In [84]:
# Top 10 most similar handshapes to 'fbx\M20241107_6263_260710_2.fbx_segment_2'

# Get the row for the specified handshape
# Get file name from index ?
index = 10

file_name = dist_mat.index[index]
handshape_row = dist_mat.iloc[index]

# Get the indices of the 10 smallest distances
handshape_distances = handshape_row[1:].to_numpy()
similar_indices = handshape_distances.argsort()

# Get the names of the 10 most similar handshapes (excluding any names starting with 'fbx\')
similar_handshape_names = dist_mat.columns[similar_indices + 1]  # +1 to account for the first column being the name
similar_handshape_names = [name for name in similar_handshape_names if not name.startswith('fbx\\')]

print("10 most similar handshapes to:")
for handshape in similar_handshape_names:
    print(handshape)

10 most similar handshapes to:
../HandformSession/HandformsTrimmed/C.FBX
../HandformSession/HandformsTrimmed/Baby_Beak_Open-2.FBX
../HandformSession/HandformsTrimmed/1_1.FBX
../HandformSession/HandformsTrimmed/B_Bent.FBX
../HandformSession/HandformsTrimmed/Q.FBX
../HandformSession/HandformsTrimmed/L.FBX
../HandformSession/HandformsTrimmed/Baby_Beak.FBX
../HandformSession/HandformsTrimmed/Baby_C.FBX
../HandformSession/HandformsTrimmed/4.FBX
../HandformSession/HandformsTrimmed/P.FBX
../HandformSession/HandformsTrimmed/T.FBX
../HandformSession/HandformsTrimmed/3.FBX
../HandformSession/HandformsTrimmed/A.FBX
../HandformSession/HandformsTrimmed/1_Curved_Jitter-3.FBX
../HandformSession/HandformsTrimmed/B.FBX
../HandformSession/HandformsTrimmed/Beak2_Open_Spread.FBX
../HandformSession/HandformsTrimmed/Baby_Beak_Open.FBX
../HandformSession/HandformsTrimmed/S.FBX
../HandformSession/HandformsTrimmed/K.FBX
../HandformSession/HandformsTrimmed/5_1.FBX
../HandformSession/HandformsTrimmed/5_Claw.FBX


In [85]:
# Get distance array
dist_mat = dist_mat.drop(columns=['Name'])
distance_array = dist_mat.to_numpy()

In [86]:
# # Make all distances 0 or higher
# distance_array += np.abs(np.min(distance_array))

In [87]:
# t-SNE on the distance matrix
random_num = random.randint(0, 10000)
# random_num = 8079  # This is the last one I used, so for reproducibility hardcoded
print("Using random seed:", random_num)
tsne_dist = manifold.TSNE(n_components=2, perplexity=50, learning_rate=200, init='random', metric='precomputed', method='barnes_hut', random_state=random_num)
X_tsne_dist = tsne_dist.fit_transform(distance_array)

Using random seed: 3701


In [88]:
# Dataframe for the px plot
tsne_df = pd.DataFrame({
    'tsne_1': X_tsne_dist[:, 0],
    'tsne_2': X_tsne_dist[:, 1],
    'name': dist_mat.columns,
    'is_handshape': [name.startswith("../") for name in dist_mat.columns]
})

# Plot
fig = px.scatter(
    tsne_df,
    x='tsne_1',
    y='tsne_2',
    color='is_handshape',
    hover_data=['name'],
    title="t-SNE Dimensionality Reduction of the Shape Features"
)

fig.update_layout(width=800, height=800)
fig.show()